# L03 Assignment — Lakeside Bank Churn + Hospital Readmissions

**⏱️ Suggested time: ~90 minutes (self-study).**

> *Sarah is loaned out to NorthStar's partner, **Lakeside Bank**. **Tom Bradley**, their Head of Analytics, has a churn problem too — but for credit-card customers. Same pipeline, different data, different feature meanings, different business cost of a false negative.*

This assignment has **two parts**:

1. **Practice (three tiers):** apply the L03 pipeline to a new domain. Each tier scaffolds less than the previous one.
2. **Independent exercises (three):** a hospital readmissions scenario where you decide everything yourself.

**Sample solutions** are at the very bottom of the notebook. *Attempt each tier and exercise before scrolling.*

**Time budget:** ~90 minutes — 60 min for practice, 30 min for exercises.

---

## Setup

In [ ]:
import numpy as np                # numpy = fast maths on lists of numbers
import pandas as pd               # pandas = tool for working with tables of data
import matplotlib.pyplot as plt   # matplotlib = tool for drawing charts
import seaborn as sns             # seaborn = prettier chart styling
import warnings                   # lets us hide harmless warning messages

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold   # splitting + cross-validation tools
from sklearn.compose import ColumnTransformer           # applies different prep to different columns
from sklearn.pipeline import Pipeline                   # chains prep steps + model into one object
from sklearn.preprocessing import StandardScaler, OneHotEncoder   # scaling numbers / turning categories into 0-1 columns
from sklearn.impute import SimpleImputer                # fills in missing values
from sklearn.linear_model import LogisticRegression     # our first classifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")   # hide harmless warnings so output stays clean
pd.set_option("display.max_columns", 20)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Setup complete.")

## Generate the Lakeside Bank dataset

We synthesise a Lakeside Bank credit-card customer file here so you don't need an extra CSV. The dataset has the same flavour as the NorthStar one — numeric and categorical features, missing values, class imbalance — but the columns mean different things (months_as_customer instead of tenure_months, card_tier instead of subscription_tier, etc.).

In [ ]:
# This cell BUILDS the practice dataset. Run it, but you do not need to study it —
# in real life this data would arrive as a CSV from the bank.
def generate_lakeside_data(n=8000, seed=2026):
    rng = np.random.default_rng(seed)

    age = rng.integers(20, 86, size=n)
    months_as_customer = rng.integers(0, 121, size=n)
    annual_income_gbp = np.round(rng.lognormal(mean=10.6, sigma=0.5, size=n).clip(15000, 250000), 0)
    avg_monthly_balance = np.round(rng.lognormal(mean=7.0, sigma=0.9, size=n).clip(0, 25000), 2)
    cards_held = rng.integers(1, 6, size=n)
    monthly_transactions = rng.poisson(lam=22, size=n).clip(0, 80)
    avg_late_payment_days = rng.exponential(scale=4.0, size=n).clip(0, 90).round(1)
    days_since_last_contact = rng.exponential(scale=45.0, size=n).clip(0, 730)

    # Missingness — some customers have no recorded contact
    missing_contact = rng.random(n) < 0.10
    contact_arr = days_since_last_contact.astype(float)
    contact_arr[missing_contact] = np.nan

    # Customer satisfaction score (analogue of L01's review_polarity)
    satisfaction = rng.normal(loc=7.0, scale=1.8, size=n).clip(1, 10)
    missing_sat = rng.random(n) < 0.25
    satisfaction_arr = satisfaction.copy()
    satisfaction_arr[missing_sat] = np.nan

    card_tier = rng.choice(["standard", "gold", "platinum"], size=n, p=[0.65, 0.25, 0.10])
    region = rng.choice(
        ["London", "Manchester", "Birmingham", "Edinburgh", "Cardiff", "Belfast"],
        size=n, p=[0.30, 0.18, 0.15, 0.15, 0.12, 0.10],
    )

    # Target — latent churn propensity (calibrated to ~15% churn rate)
    logit = (
        -0.6
        + (-0.018 * months_as_customer)
        + (0.06  * avg_late_payment_days)
        + (-0.30 * np.nan_to_num(satisfaction_arr, nan=7.0))
        + (0.006 * np.nan_to_num(contact_arr, nan=45.0))
        + (-0.5  * (card_tier == "platinum").astype(float))
        + (-0.2  * (card_tier == "gold").astype(float))
        + (0.4   * (cards_held == 1).astype(float))
    )
    prob = 1 / (1 + np.exp(-logit))
    churned = (rng.random(n) < prob).astype(int)

    return pd.DataFrame({
        "customer_id":            np.arange(200000, 200000 + n),
        "age":                    age,
        "months_as_customer":     months_as_customer,
        "annual_income_gbp":      annual_income_gbp,
        "avg_monthly_balance":    avg_monthly_balance,
        "cards_held":             cards_held,
        "monthly_transactions":   monthly_transactions,
        "avg_late_payment_days":  avg_late_payment_days,
        "days_since_last_contact": contact_arr,
        "customer_satisfaction":  satisfaction_arr,
        "card_tier":              card_tier,
        "region":                 region,
        "churned":                churned,
    })


lakeside = generate_lakeside_data()
print(f"Lakeside dataset: {len(lakeside):,} rows × {lakeside.shape[1]} columns")
print(f"Churn rate: {lakeside['churned'].mean():.1%}")
print(f"Missing in days_since_last_contact: {lakeside['days_since_last_contact'].isna().mean():.1%}")
print(f"Missing in customer_satisfaction:   {lakeside['customer_satisfaction'].isna().mean():.1%}")
print()
lakeside.head()

---

## 📚 Choose your track

This assignment has **two tracks**. Pick **one** based on your background — you don't need to do both.

| Track | Who it's for | What you'll do |
|---|---|---|
| **🟢 Foundational Track** | Learners new to ML / programming | Tier 1 (worked LR pipeline) + Tier 2 (k-fold fill-in) |
| **🔵 Advanced Track** | Learners with prior ML background | Tier 3 (open capacity threshold) + Part B hospital exercises |

**Minimum viable submission:** if you're short on time, completing the **Foundational Track on the bank (churn) dataset only** is a full, acceptable submission — the hospital exercises in Part B are for the Advanced Track or extra practice.

If you're unsure, start with the **Foundational Track**. If it feels easy, skip ahead to the **Advanced Track** — both tracks cover the same lesson outcomes; only the scaffolding differs.

---


---

# 🟢 Foundational Track

> *No prior ML background needed. The cells below are scaffolded — read the worked example, then fill in the blanks. Hints are included.*

---


# Part A — Practice (three tiers)

Each tier scaffolds less than the previous one. Work through them in order.

---

## 🟢 Tier 1 — Guided

Below is a complete, working pipeline applied to the Lakeside data. Read it, run it, and **answer the reflection questions at the end**.

In [ ]:
# === Tier 1 — Fully worked example ===

y_L = lakeside["churned"]
X_L = lakeside.drop(columns=["customer_id", "churned"])

# Hold back 20% as an unseen test set; stratify keeps the churn rate equal in both
X_train, X_test, y_train, y_test = train_test_split(
    X_L, y_L, test_size=0.20, stratify=y_L, random_state=42,
)

numeric_features_L = [
    "age", "months_as_customer", "annual_income_gbp",
    "avg_monthly_balance", "cards_held", "monthly_transactions",
    "avg_late_payment_days", "days_since_last_contact", "customer_satisfaction",
]
categorical_features_L = ["card_tier", "region"]

preprocessor_L = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features_L),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features_L),
])

pipe_L = Pipeline([("prep", preprocessor_L),
                   ("model", LogisticRegression(max_iter=1000, random_state=42))])
pipe_L.fit(X_train, y_train)

# Predict and report at default 0.5
y_proba_L = pipe_L.predict_proba(X_test)[:, 1]
y_pred_L  = (y_proba_L >= 0.5).astype(int)

print(f"Test accuracy:  {accuracy_score(y_test, y_pred_L):.3f}")
print(f"Test precision: {precision_score(y_test, y_pred_L, zero_division=0):.3f}")
print(f"Test recall:    {recall_score(y_test, y_pred_L, zero_division=0):.3f}")
print(f"Test F1:        {f1_score(y_test, y_pred_L, zero_division=0):.3f}")

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_L,
                                         display_labels=["stayed", "churn"])
plt.title("Lakeside Bank — confusion matrix @ threshold 0.5")
plt.tight_layout()
plt.show()

### Tier 1 reflection (3 questions)

*Double-click and answer below. Sample answers at the bottom of the notebook.*

1. What's striking about this confusion matrix compared with the NorthStar one from notebook 04?
2. If you lowered the threshold from 0.5 to 0.3, would precision go up or down? Would recall go up or down?
3. Look at the column `cards_held`. Does carrying more cards predict churn or retention? (Make a 1-sentence hypothesis without running any new code.)

*Your answers:*

1.
2.
3.

## 🟡 Tier 2 — Partial (fill in the blanks)

The skeleton below trains the same pipeline but uses **k-fold cross-validation**. Fill in the four blanks marked `___`. Hints in HTML `<details>` sections (click to expand).

In [ ]:
# === Tier 2 — Fill in the blanks (look for ___) ===

# Set up the cross-validator
# Hint: 5-fold, stratified, shuffled with random_state=42
cv = None    # ← fill in

# Cross-validated accuracy of the pipeline already built above (pipe_L)
# Hint: pass pipe_L, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1
cv_scores = None    # ← fill in

if cv is not None and cv_scores is not None:
    print(f"CV accuracy scores: {[f'{s:.3f}' for s in cv_scores]}")
    print(f"Mean: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Bonus — what if we cared about recall, not accuracy?
# Set scoring='recall' below to see how the model does on catching real churners
# Hint: scoring="recall"
cv_recall_scores = None    # ← fill in
if cv_recall_scores is not None:
    print(f"\nCV recall scores: {[f'{s:.3f}' for s in cv_recall_scores]}")
    print(f"Mean: {cv_recall_scores.mean():.3f} ± {cv_recall_scores.std():.3f}")

<details>
<summary>💡 Hint for the first blank</summary>

```python
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
```
</details>

<details>
<summary>💡 Hint for the second blank</summary>

```python
cv_scores = cross_val_score(pipe_L, X_train, y_train, cv=cv,
                            scoring="accuracy", n_jobs=-1)
```
</details>

<details>
<summary>💡 Hint for the third blank</summary>

```python
cv_recall_scores = cross_val_score(pipe_L, X_train, y_train, cv=cv,
                                    scoring="recall", n_jobs=-1)
```
</details>

### Tier 2 reflection (2 questions)

1. The CV recall is much lower than the CV accuracy. Why?
2. If you needed recall above 0.5, what would you change in the pipeline?

*Your answers:*

1.
2.

---

# 🔵 Advanced Track

> *For learners with prior ML background. Minimal scaffolding — you decide the approach. You're welcome to peek at the Foundational Track above for reference.*

---


## 🟠 Tier 3 — Open

A scenario, a target, and explicit success criteria. The notebook gives you no code scaffold.

> **Scenario.** Tom Bradley wants a list of customers to call for retention this month. His team can handle **300 customers** from the 1,600 in the Lakeside test set. He wants the model to surface those 300 customers, prioritising the ones most likely to actually churn.
>
> **Your job:**
>
> 1. From the trained `pipe_L`, get predicted churn probabilities on `X_test`.
> 2. Find the threshold that flags the top 300 customers (ranked by probability).
> 3. Compute precision, recall, and F1 at that threshold.
> 4. Report — in plain English — how many real churners Tom's team will catch.
>
> **Success criteria:**
>
> - Threshold found programmatically (not hardcoded to 0.5).
> - Number flagged equals 300 ± 5.
> - Final printout includes: threshold, n flagged, precision, recall, F1, "Tom's team will catch X out of Y real churners."

Write your code in the cell below.

In [ ]:
# === Tier 3 — Your open-ended code goes here ===

# Steps to follow:
# 1. Compute y_proba_L (already done above) — predicted churn probabilities.
# 2. Sort the probabilities descending. The 300th value IS your threshold.
# 3. Apply that threshold, compute the metrics, print the report.

# (write your answer here)


# Part B — Independent exercises (hospital readmissions)

A different domain to test that you can transfer the pipeline.

> **Scenario.** A hospital wants to predict which patients admitted to its cardiology ward will be readmitted within 30 days of discharge. Reducing 30-day readmissions saves money and improves outcomes.
>
> Three exercises follow. Each comes with explicit tasks and a blank code cell. **Try each on your own. Sample solutions at the bottom.**

In [ ]:
# Generate a synthetic hospital readmission dataset
# This cell BUILDS the hospital dataset. Run it; no need to study the generator.
def generate_hospital_data(n=4000, seed=99):
    rng = np.random.default_rng(seed)
    age                = rng.integers(20, 95, size=n)
    length_of_stay     = rng.poisson(lam=5, size=n).clip(1, 30)
    num_diagnoses      = rng.integers(1, 12, size=n)
    num_meds           = rng.integers(0, 25, size=n)
    had_prior          = rng.integers(0, 2, size=n)
    bmi                = np.round(rng.normal(28, 6, size=n).clip(15, 55), 1)
    discharge_to       = rng.choice(["home", "rehab", "snf", "home_health"], size=n,
                                     p=[0.55, 0.15, 0.20, 0.10])
    primary_diagnosis  = rng.choice(["heart_failure", "myocardial_infarction",
                                       "arrhythmia", "valve_disease", "other"],
                                       size=n, p=[0.30, 0.25, 0.20, 0.15, 0.10])

    # Causal structure for readmission (intercept calibrated to ~18% base rate)
    logit = (
        -6.0
        + (0.025 * age)
        + (0.10  * length_of_stay)
        + (0.18  * num_diagnoses)
        + (1.1   * had_prior)
        + (0.06  * (bmi - 28))
        + (0.5   * (primary_diagnosis == "heart_failure").astype(float))
        + (-0.4  * (discharge_to == "home").astype(float))
        + (-0.3  * (discharge_to == "home_health").astype(float))
    )
    prob = 1 / (1 + np.exp(-logit))
    readmitted = (rng.random(n) < prob).astype(int)

    return pd.DataFrame({
        "patient_id":          np.arange(300000, 300000 + n),
        "age":                 age,
        "length_of_stay":      length_of_stay,
        "num_diagnoses":       num_diagnoses,
        "num_meds_at_discharge": num_meds,
        "had_prior_admission": had_prior,
        "bmi":                 bmi,
        "discharge_to":        discharge_to,
        "primary_diagnosis":   primary_diagnosis,
        "readmitted_30d":      readmitted,
    })

hosp = generate_hospital_data()
print(f"Hospital dataset: {len(hosp):,} rows · readmission rate: {hosp['readmitted_30d'].mean():.1%}")
hosp.head()

### Exercise 1 — Build the pipeline

**Tasks:**
1. Split features (`X_H`) and target (`y_H` = `readmitted_30d`). Drop `patient_id`.
2. Train/test split (80/20), stratify on the target.
3. Identify numerical and categorical features.
4. Build a `ColumnTransformer` (median impute + standard scale | mode impute + one-hot encode).
5. Wrap with a `LogisticRegression(max_iter=1000)` in a single `Pipeline`.
6. Fit on train; print test accuracy.

*Your code:*

In [ ]:
# Exercise 1 — pipeline build
# (your answer here)


*Your interpretation (1–2 sentences): is the test accuracy meaningful here, given the 18% positive class rate?*

> (your answer here)

### Exercise 2 — Evaluate with cross-validation and the right metric

**Tasks:**
1. Set up a 5-fold stratified cross-validator with `random_state=42`.
2. Compute CV `accuracy`, `precision`, and `recall` on the training set.
3. Print all three with their standard deviations.

*Your code:*

In [ ]:
# Exercise 2 — CV evaluation
# (your answer here)


*Your interpretation: which of accuracy, precision, recall is most useful here?*

> (your answer here)

### Exercise 3 — Threshold for a 50-bed cardiology rehab capacity

**Scenario.** The hospital's cardiology rehab can take 50 patients per quarter from this test set (out of 800 test patients). They want the model to surface the 50 highest-risk patients.

**Tasks:**
1. Get probabilities from your trained pipeline on `X_test_H`.
2. Find the threshold that flags exactly 50 (or 50 ± 2) patients.
3. Compute precision and recall at that threshold.
4. Print a one-sentence recommendation: how many real readmitters will be in the rehab cohort?

*Your code:*

In [ ]:
# Exercise 3 — threshold by capacity
# (your answer here)


*Your final recommendation to the hospital (2 sentences):*

> (your answer here)

## ✅ Submission checklist

Before submitting (or before checking the sample solutions):

- [ ] All three Tier 1 reflection questions answered
- [ ] Tier 2 blanks filled in and CV scores printed (both accuracy and recall)
- [ ] Tier 2 reflection questions answered
- [ ] Tier 3 code prints threshold, n flagged ≈ 300, precision, recall, F1
- [ ] Exercise 1: working pipeline + test accuracy + interpretation
- [ ] Exercise 2: CV accuracy, precision, recall + interpretation
- [ ] Exercise 3: threshold + final recommendation

---

# 📚 Sample solutions

*Compare yours to these only after attempting each.*

## Sample — Tier 1 reflection

1. **Striking difference:** the Lakeside model probably catches more churners at the default threshold than the NorthStar model did. Likely because the Lakeside churn rate is higher, the model's probability outputs span a wider range, and 0.5 lands somewhere more useful. (Run the cell to confirm with your own numbers.)
2. **Threshold drop 0.5 → 0.3:** Precision DOWN (more false positives), Recall UP (catches more real churners).
3. **`cards_held`:** a customer with only ONE card is more committed (only relationship with the bank). A customer with several cards has more options and might be shopping around. Hypothesis: more cards → higher churn risk.

## Sample — Tier 2 solution

In [ ]:
# === Tier 2 — Sample solution ===

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(pipe_L, X_train, y_train, cv=cv,
                             scoring="accuracy", n_jobs=-1)
print(f"CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

cv_recall_scores = cross_val_score(pipe_L, X_train, y_train, cv=cv,
                                    scoring="recall", n_jobs=-1)
print(f"CV recall:   {cv_recall_scores.mean():.3f} ± {cv_recall_scores.std():.3f}")

**Tier 2 reflection answers:**

1. Recall is much lower than accuracy because the dataset is imbalanced and the default 0.5 threshold catches only the very-high-probability cases. Accuracy is dominated by correctly predicting "stayed" for the majority class.
2. To raise recall: lower the threshold (next exercise), use `class_weight='balanced'` in `LogisticRegression`, or use a different scoring metric in your CV.

## Sample — Tier 3 solution

In [ ]:
# === Tier 3 — Sample solution ===

n_to_flag = 300
# Sort probabilities; the 300th-largest value becomes the cut-off
threshold = np.sort(y_proba_L)[-n_to_flag]   # the (n_to_flag)-th largest probability
y_pred_top = (y_proba_L >= threshold).astype(int)

p = precision_score(y_test, y_pred_top, zero_division=0)
r = recall_score(y_test, y_pred_top, zero_division=0)
f = f1_score(y_test, y_pred_top, zero_division=0)
flagged   = int(y_pred_top.sum())
real_caught = int(((y_pred_top == 1) & (y_test == 1)).sum())
total_churners = int(y_test.sum())

print(f"Threshold (capacity-based):  {threshold:.3f}")
print(f"Customers flagged:           {flagged}")
print(f"Precision:                   {p:.3f}")
print(f"Recall:                      {r:.3f}")
print(f"F1:                          {f:.3f}")
print()
print(f"→ Tom's retention team will catch {real_caught} out of {total_churners} real churners.")
print(f"   The other {flagged - real_caught} calls are to customers who would have stayed anyway —")
print(f"   that's the operational cost of running an early-warning model at capacity.")

## Sample — Exercise 1 solution

In [ ]:
# === Exercise 1 — Sample solution ===

y_H = hosp["readmitted_30d"]
X_H = hosp.drop(columns=["patient_id", "readmitted_30d"])

# Same recipe as the bank data: split, keeping the readmission rate equal in both halves
X_train_H, X_test_H, y_train_H, y_test_H = train_test_split(
    X_H, y_H, test_size=0.20, stratify=y_H, random_state=42,
)

numeric_features_H = ["age", "length_of_stay", "num_diagnoses", "num_meds_at_discharge",
                       "had_prior_admission", "bmi"]
categorical_features_H = ["discharge_to", "primary_diagnosis"]

preprocessor_H = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features_H),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features_H),
])

pipe_H = Pipeline([("prep", preprocessor_H),
                   ("model", LogisticRegression(max_iter=1000, random_state=42))])
pipe_H.fit(X_train_H, y_train_H)
print(f"Test accuracy: {pipe_H.score(X_test_H, y_test_H):.3f}")
print(f"Baseline (always 'not readmitted'): {1 - y_test_H.mean():.3f}")

**Interpretation:** test accuracy will be very close to the baseline of 82% (since `readmitted_30d` is ~18% positive). On its own that number doesn't say whether the model is learning anything — we need precision/recall (Exercise 2).

## Sample — Exercise 2 solution

In [ ]:
# === Exercise 2 — Sample solution ===

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for metric_name in ["accuracy", "precision", "recall"]:
    scores = cross_val_score(pipe_H, X_train_H, y_train_H, cv=cv,
                              scoring=metric_name, n_jobs=-1)
    print(f"CV {metric_name:10s}: {scores.mean():.3f} ± {scores.std():.3f}")

**Interpretation:** for a hospital readmission model, **recall** is most important. A false negative is a patient who comes back to the ER in a critical condition. A false positive (placing a patient in rehab who didn't strictly need it) is an operational cost but not a clinical risk.

## Sample — Exercise 3 solution

In [ ]:
# === Exercise 3 — Sample solution ===

y_proba_H = pipe_H.predict_proba(X_test_H)[:, 1]

n_to_flag = 50
# Capacity-based threshold: the 50th-largest probability becomes the cut-off
threshold = np.sort(y_proba_H)[-n_to_flag]
y_pred_H = (y_proba_H >= threshold).astype(int)

n_flagged = int(y_pred_H.sum())
real_caught = int(((y_pred_H == 1) & (y_test_H == 1)).sum())
total_readmits = int(y_test_H.sum())

precision_H = precision_score(y_test_H, y_pred_H, zero_division=0)
recall_H    = recall_score(y_test_H, y_pred_H, zero_division=0)

print(f"Threshold:   {threshold:.3f}")
print(f"Flagged:     {n_flagged}")
print(f"Precision:   {precision_H:.3f}")
print(f"Recall:      {recall_H:.3f}")
print()
print(f"Recommendation to the hospital:")
print(f"  Of the {n_flagged} highest-risk patients flagged, approximately {real_caught} will")
print(f"  actually be readmitted within 30 days — out of {total_readmits} total in this test set.")
print(f"  That captures about {recall_H:.0%} of all readmissions at a capacity of {n_flagged} rehab beds.")

## What's next

You've now applied the L03 pipeline to two new domains. The pattern transfers cleanly:

- *Same preprocessing.* Median impute + standard scale + mode impute + one-hot encode.
- *Same model.* Logistic regression baseline.
- *Same evaluation discipline.* Confusion matrix → precision/recall/F1 → business-aware threshold.
- *Different cost trade-offs.* In banking, the FN cost (lost lifetime value) is high. In healthcare, the FN cost (a readmission) is even higher.

**Next session → L04 (Advanced Supervised Learning).** Sarah's question — *"can I do better than logistic regression?"* — gets its answer: decision trees, random forests, and gradient-boosted ensembles. The same pipeline you built here just swaps the final step.

If you want to go deeper on the theory underneath L03 — bias-variance math, ROC-AUC, learning curves — open `optional_extensions.ipynb`.